In [1]:
!pip install -q -U transformers peft accelerate datasets bitsandbytes trl

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

print(f"Loading {MODEL_ID}...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

print(f"{MODEL_ID} loaded on {model.device}")
print(f" Vocab size: {len(tokenizer)}")

C:\Users\PAVISHANTH\Desktop\DSGP\Implementation\.venv2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading Qwen/Qwen2.5-3B-Instruct...


C:\Users\PAVISHANTH\Desktop\DSGP\Implementation\.venv2\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\PAVISHANTH\.cache\huggingface\hub\models--Qwen--Qwen2.5-3B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 

Qwen/Qwen2.5-3B-Instruct loaded on cuda:0
 Vocab size: 151665


In [3]:
import torch

def test_qwen_model(user_input, max_new_tokens=150):
    """Test Qwen model with Sinhala input"""

    # Format using Qwen's chat template
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": user_input}
    ]

    # Apply chat template
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Tokenize
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode (remove input from output)
    response_ids = outputs[0][len(inputs.input_ids[0]):]
    response = tokenizer.decode(response_ids, skip_special_tokens=True)

    return response


# Test with Sinhala
print("="*70)
print("TESTING QWEN WITH SINHALA")
print("="*70)

test_prompts = [
    "කොහොමද ඔයාට?",  # How are you?
    "ශ්‍රී ලංකාවේ අගනුවර කුමක්ද?",  # What is the capital of Sri Lanka?
    "මට දුක හිතෙනවා",  # I feel sad
]

for i, prompt in enumerate(test_prompts, 1):
    print(f"\n{i}. Input: {prompt}")
    try:
        response = test_qwen_model(prompt)
        print(f"   Response: {response[:200]}...")
    except Exception as e:
        print(f"   Error: {str(e)[:100]}")

print("\n Base model testing complete!")

TESTING QWEN WITH SINHALA

1. Input: කොහොමද ඔයාට?
   Response: යුතු ට්‍රෝට් කොහෙ යලේ ඔයාට් (Coyote) වන අඩුකළේ නාගර්ජයකින් අයිලුවකි. ඔයාට් යුතු බෝධ හීපරු කෙටි කැම්බේ නාගර...

2. Input: ශ්‍රී ලංකාවේ අගනුවර කුමක්ද?
   Response: යිහි, ලංකාවේ අගනුවර වසුනු කණ්ඩායම ලංකා නාගරිකයාවා සිටියන්නේ ගණනාතම ප්‍රකාශයෙන් අගනුවර සියල්ලේ. මෙම අගනුව...

3. Input: මට දුක හිතෙනවා
   Response: මෙම ප්‍රයෝගේ කො:view කරුණු කුලුවකු ඇති මුදලු නිර්දේශයකු හැකියුණු හැකියුණු වුණෙලුමු. නිදහස් ලේඛකයන් යුතු ක�...

 Base model testing complete!


In [9]:
import pandas as pd
from datasets import Dataset

# ✅ CORRECT PATH (remove duplicate "DATA_PATH =")
DATA_PATH = "C:/Users/PAVISHANTH/Desktop/DSGP/Implementation/E.motion-/processed_sinhala/train_si.jsonl"

print(f"Loading Sinhala dataset from: {DATA_PATH}")

# Load JSONL
df = pd.read_json(DATA_PATH, lines=True)

print(f"Loaded {len(df)} training samples")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nSample data:")
print(df.head(2))

# Show a sample prompt
print("\n" + "="*70)
print("SAMPLE PROMPT:")
print("="*70)
print(df['prompt'].iloc[0])
print("\nTARGET:")
print(df['target'].iloc[0])

Loading Sinhala dataset from: C:/Users/PAVISHANTH/Desktop/DSGP/Implementation/E.motion-/processed_sinhala/train_si.jsonl
Loaded 796 training samples

Columns: ['prompt', 'target', 'language', 'user_emotion']

Sample data:
                                              prompt  \
0  Context: The user is feeling neutral. Task: Re...   
1  Context: The user is feeling disgust. Task: Re...   

                                              target language user_emotion  
0  ඩවුන්ලෝඩ්ස් අවශ්‍යතා හෝ රුචිකත්වයන් පිළිබිඹු ක...       si      neutral  
1  සමාජ වැරදි පියවර සෑම කෙනෙකුටම සිදු වේ. ඒවා ඉගෙ...       si      disgust  

SAMPLE PROMPT:
Context: The user is feeling neutral. Task: Respond as an empathetic assistant.
User: මම මගේ මෑත කාලීන ඩවුන්ලෝඩ්ස් පරීක්ෂා කරමින් සිටිනවා
Assistant:

TARGET:
ඩවුන්ලෝඩ්ස් අවශ්‍යතා හෝ රුචිකත්වයන් පිළිබිඹු කරයි. ඒවා ඔබගේ වර්තමාන ප්‍රමුඛතා පිළිබඳව හෙළි කරන්නේ කුමක්ද?


In [10]:
def format_for_qwen(example):
    """
    Convert your data format to Qwen's chat format

    Your format:
    - prompt: "Context: The user is feeling දුක. Task: Respond...\nUser: මට දුක හිතෙනවා\nAssistant:"
    - target: "ඔබට දුක හිතෙන බව..."

    Qwen format:
    - "<|im_start|>user\n{user_input}<|im_end|>\n<|im_start|>assistant\n{response}<|im_end|>"
    """

    # Extract just the user input from your prompt
    prompt_text = example['prompt']

    # Parse your prompt format to get the user input
    if "User:" in prompt_text and "Assistant:" in prompt_text:
        user_part = prompt_text.split("User:")[1].split("Assistant:")[0].strip()
    else:
        user_part = prompt_text

    # Format for Qwen
    text = (
        f"<|im_start|>system\n"
        f"You are an empathetic counselor and therapist. Respond in Sinhala.<|im_end|>\n"
        f"<|im_start|>user\n{user_part}<|im_end|>\n"
        f"<|im_start|>assistant\n{example['target']}<|im_end|>"
    )

    return {"text": text}

# Apply formatting
print("Formatting dataset for Qwen...")
dataset = Dataset.from_pandas(df)
formatted_dataset = dataset.map(format_for_qwen)

print(f" Formatted {len(formatted_dataset)} samples")

# Show formatted sample
print("\n" + "="*70)
print("FORMATTED SAMPLE (ready for training):")
print("="*70)
print(formatted_dataset[0]['text'])

Formatting dataset for Qwen...


Map: 100%|██████████| 796/796 [00:00<00:00, 12940.32 examples/s]

 Formatted 796 samples

FORMATTED SAMPLE (ready for training):
<|im_start|>system
You are an empathetic counselor and therapist. Respond in Sinhala.<|im_end|>
<|im_start|>user
මම මගේ මෑත කාලීන ඩවුන්ලෝඩ්ස් පරීක්ෂා කරමින් සිටිනවා<|im_end|>
<|im_start|>assistant
ඩවුන්ලෝඩ්ස් අවශ්‍යතා හෝ රුචිකත්වයන් පිළිබිඹු කරයි. ඒවා ඔබගේ වර්තමාන ප්‍රමුඛතා පිළිබඳව හෙළි කරන්නේ කුමක්ද?<|im_end|>


In [12]:
# Load validation data
VAL_PATH = "C:/Users/PAVISHANTH/Desktop/DSGP/Implementation/E.motion-/processed_sinhala/val_si.jsonl"

print(f"Loading validation dataset from: {VAL_PATH}")

val_df = pd.read_json(VAL_PATH, lines=True)
val_dataset = Dataset.from_pandas(val_df)
formatted_val = val_dataset.map(format_for_qwen)

print(f"Loaded {len(formatted_val)} validation samples")

Loading validation dataset from: C:/Users/PAVISHANTH/Desktop/DSGP/Implementation/E.motion-/processed_sinhala/val_si.jsonl


Map: 100%|██████████| 99/99 [00:00<00:00, 12131.12 examples/s]

Loaded 99 validation samples


In [13]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

print("Configuring LoRA...")

lora_config = LoraConfig(
    r=16,  # Rank
    lora_alpha=32,  # Alpha (2x rank is good)
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # Attention layers
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Prepare model for training
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)

print("\n" + "="*70)
print("TRAINABLE PARAMETERS:")
print("="*70)
model.print_trainable_parameters()

Configuring LoRA...

TRAINABLE PARAMETERS:
trainable params: 7,372,800 || all params: 3,093,311,488 || trainable%: 0.2383


In [15]:
from trl import SFTConfig

training_config = SFTConfig(
    output_dir="./qwen-sinhala-therapy",

    # Training hyperparameters
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,

    # Learning rate
    learning_rate=2e-4,
    warmup_steps=50,

    # Optimization
    optim="paged_adamw_8bit",

    # Logging and saving
    logging_steps=10,
    save_steps=100,
    eval_steps=50,
    eval_strategy="steps",

    # Memory
    fp16=True,
    max_length=512,

    # Dataset
    dataset_text_field="text",
    packing=False,

    # Save best
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)

print("Training configuration updated successfully!")

Training configuration updated successfully!


In [19]:
# 1. Verify and fix the column name
def ensure_text_column(example):
    return {"text": example["text"]}

formatted_dataset = formatted_dataset.map(ensure_text_column)
formatted_val = formatted_val.map(ensure_text_column)

formatted_dataset = formatted_dataset.remove_columns([c for c in formatted_dataset.column_names if c != 'text'])
formatted_val = formatted_val.remove_columns([c for c in formatted_val.column_names if c != 'text'])

print(f" Dataset columns: {formatted_dataset.column_names}")

Map: 100%|██████████| 99/99 [00:00<00:00, 24165.52 examples/s]

 Dataset columns: ['text']


In [21]:
from trl import SFTConfig, SFTTrainer
import torch

# 1. Training configuration
training_config = SFTConfig(
    output_dir="./qwen-sinhala-therapy",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    warmup_steps=50,
    optim="paged_adamw_8bit",
    logging_steps=10,
    save_steps=100,
    eval_steps=50,
    eval_strategy="steps",

    fp16=False,
    bf16=False,

    max_length=512,
    dataset_text_field="text",
    gradient_checkpointing=True,
    report_to="none"
)

# 2. Create Trainer
trainer = SFTTrainer(
    model=model,
    args=training_config,
    train_dataset=formatted_dataset,
    eval_dataset=formatted_val,
    processing_class=tokenizer,
)

print("Starting training in Native Float16 Mode...")

# 3. Execution
torch.cuda.empty_cache()
try:
    trainer.train()
    print("\nTRAINING COMPLETE!")
except Exception as e:
    print(f"\nFinal Error Check: {e}")

Truncating eval dataset: 100%|██████████| 99/99 [00:00<00:00, 47569.72 examples/s]


Starting training in Native Float16 Mode...


Step,Training Loss,Validation Loss
50,1.111820,1.070395
100,0.946827,0.916275
150,0.846000,0.851057
200,0.832544,0.818011
250,0.776225,0.798729
300,0.742198,0.790071



TRAINING COMPLETE!


In [22]:
# Save the fine-tuned model
SAVE_PATH = "./qwen-sinhala-therapy-final"

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print(f" Model saved to {SAVE_PATH}")

 Model saved to ./qwen-sinhala-therapy-final


In [23]:
def test_finetuned_qwen(user_input, max_tokens=150):
    """Test the fine-tuned Qwen model"""

    messages = [
        {"role": "system", "content": "You are an empathetic counselor and therapist. Respond in Sinhala."},
        {"role": "user", "content": user_input}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.8,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.15,
            pad_token_id=tokenizer.eos_token_id
        )

    response_ids = outputs[0][len(inputs.input_ids[0]):]
    response = tokenizer.decode(response_ids, skip_special_tokens=True)

    return response


print("="*70)
print("TESTING FINE-TUNED QWEN MODEL")
print("="*70)

test_cases = [
    "මට දුක හිතෙනවා",
    "මට කාංසාව දැනෙනවා",
    "මම හුදකලා දැනෙනවා",
    "මට කෝපයක් දැනෙනවා",
]

for i, test in enumerate(test_cases, 1):
    print(f"\n{'='*70}")
    print(f"TEST {i}/{len(test_cases)}")
    print(f"{'='*70}")
    print(f"User: {test}")

    try:
        response = test_finetuned_qwen(test, max_tokens=150)
        print(f"Counselor: {response}")

        # Quality check
        words = len(response.split())
        has_sinhala = any('\u0D80' <= char <= '\u0DFF' for char in response)

        print(f"\nQuality: {words} words | Sinhala: {'✅' if has_sinhala else '❌'}")

    except Exception as e:
        print(f"Error: {str(e)[:100]}")

print("\n FINE-TUNED MODEL TESTING COMPLETE!")

TESTING FINE-TUNED QWEN MODEL

TEST 1/4
User: මට දුක හිතෙනවා
Counselor: ඩසය්ගලීද කරපතේ - අහඟචාණෝ, ඇතිබූමැනි ගෘතම-අගය පිළිකුල් වීම කෙරෙහි සම්බන්ධ වේ. හඳුනා ගැන මෙච්චර ෆෑරිල් තමයක�

Quality: 16 words | Sinhala: ✅

TEST 2/4
User: මට කාංසාව දැනෙනවා
Counselor: කාංසාව පරීක්ළ වූයේ එකොදිනුම්, යගලකත්‍රණ ඇතිබාහිනි ලෙස නැවත උපකාර කළ හැකිය. මොන කෝපයක් ආච්චිම කරනවා?

Quality: 15 words | Sinhala: ✅

TEST 3/4
User: මම හුදකලා දැනෙනවා
Counselor: හූටත්ගරසියොඳීම, කළපෝර සබාධේන තියා එය පෙන්වනවා - මිතුර් වශයෙන්ම ලෙස යථා ඇදී නොන෠ච්චරය ආරක්ෂණය කිරීමට උත්සාහ

Quality: 16 words | Sinhala: ✅

TEST 4/4
User: මට කෝපයක් දැනෙනවා
Counselor: "ඒක වගේ හරහස සතුදී මොනවා වූ සාධන කළ හැක." නිවල පෑචිල් හිබීමෙන්, යථාජු ශක්තිය තොරව දැනෙනවා. ෆෛෂණික චිත්�

Quality: 18 words | Sinhala: ✅

 FINE-TUNED MODEL TESTING COMPLETE!
